# 手写数字识别实验入口

这个 notebook 用于分阶段运行：参数检查、可选数据准备、HPO、clean 训练、robust 微调、模型评估、ensemble 权重搜索、文件夹预测与 preprocess debug。默认只做评估，不会自动训练。

In [6]:
from pathlib import Path
from datetime import datetime
import importlib
import json
import sys
import time

import pandas as pd
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path(r"E:\\ALL\\学习\\AI导论作业-识别手写数字")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config as config_module
import src.data as data_module
import src.download_finetune_data as download_finetune_data
import src.engine as engine_module
import src.ensemble_predict as ensemble_predict
import src.evaluate as evaluate_module
import src.hpo as hpo_module
import src.model as model_module
import src.robust_data as robust_data_module
import src.robust_train as robust_train
import src.validation_board as validation_board

for module in [
    config_module,
    data_module,
    download_finetune_data,
    engine_module,
    ensemble_predict,
    evaluate_module,
    hpo_module,
    model_module,
    robust_data_module,
    robust_train,
    validation_board,
]:
    importlib.reload(module)

from src.config import ExperimentConfig, ensure_project_paths
from src.data import create_dataloaders
from src.download_finetune_data import prepare_all as prepare_finetune_datasets
from src.engine import fit
from src.ensemble_predict import predict_image_folder, search_ensemble_weight, predict_batch_pair
from src.evaluate import evaluate_external_holdouts, evaluate_mnist_c_zip, load_model_from_checkpoint
from src.hpo import run_hpo
from src.model import build_model, count_model_parameters
from src.predict import PredictionImageDataset, save_preprocess_debug_visualization, write_predictions_csv
from src.robust_train import run_robust_finetune
from src.train import set_seed
from src.validation_board import evaluate_validation_board

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
{"project_root": str(PROJECT_ROOT), "device": DEVICE, "cuda_available": torch.cuda.is_available()}

{'project_root': 'e:\\ALL\\学习\\AI导论作业-识别手写数字',
 'device': 'cuda',
 'cuda_available': True}

## 1. 运行开关与可调参数

默认只评估已有 checkpoint。需要训练、调参、预测时，只打开对应开关。

In [7]:
DO_PREPARE_DATA = False
DO_HPO = False
DO_TRAIN_CLEAN = False
DO_TRAIN_ROBUST = True
DO_EVALUATE = True
DO_ENSEMBLE_SEARCH = True
DO_PREDICT_FOLDER = False

if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

PATH_CONFIG = {
    "output_dir": PROJECT_ROOT / "outputs_submission",
    "exam_image_dir": PROJECT_ROOT / "exam_data" / "test",
}

CHECKPOINT_CANDIDATES = [
    PATH_CONFIG["output_dir"] / "checkpoints" / "best_model_stat-09987e.pt",
    PATH_CONFIG["output_dir"] / "checkpoints" / "checkpoint_clean_best.pth",
    PATH_CONFIG["output_dir"] / "checkpoints" / "best_model_state_09974.pt",
    PATH_CONFIG["output_dir"] / "checkpoints" / "best_model_state.pt",
]
CLEAN_CHECKPOINT = next((path for path in CHECKPOINT_CANDIDATES if path.exists()), CHECKPOINT_CANDIDATES[-1])
ROBUST_CHECKPOINT = PATH_CONFIG["output_dir"] / "checkpoints" / "robust_expert_best.pt"

DATA_CONFIG = {
    "dataset_name": "multisource",
    "use_mnist": True,
    "use_emnist_digits": True,
    "use_usps": True,
    "use_qmnist": True,
    "emnist_max_samples": 50000,
    "qmnist_max_samples": 60000,
    "external_holdout_names": ("mnist_test", "emnist_digits_test", "qmnist_test10k"),
}
MODEL_CONFIG = {"model_name": "medium_cnn", "dropout": 0.21672530847241062}
TRAIN_CONFIG = {
    "batch_size": 4096,
    "epochs": 60,
    "learning_rate": 0.0008398721379146775,
    "optimizer_type": "AdamW",
    "scheduler_type": "CosineAnnealingLR",
    "weight_decay": 6.602542933207749e-06,
    "label_smoothing": 0.03,
    "num_workers": 12,
    "pin_memory": True,
    "persistent_workers": True,
    "prefetch_factor": 12,
    "dataloader_timeout": 0,
    "use_amp": True,
    "allow_tf32": True,
    "compile_model": False,
    "compile_mode": "max-autotune",
    "use_early_stopping": True,
    "early_stopping_patience": 10,
}
AUGMENT_CONFIG = {
    "rotation_degrees": 7.536974266650085,
    "translate_ratio": 0.05567431908414762,
    "scale_min": 0.9213414099692713,
    "scale_max": 1.1010803603468335,
    "shear_degrees": 4.9755817645420075,
    "use_random_affine": True,
    "use_gaussian_blur": False,
}
ROBUST_CONFIG = {
    "fine_tune_lr": 3e-5,
    "fine_tune_epochs": 30,
    "batch_size": 4096,
    "robust_aug_strength": "strong",
    "mnist_family_weight": 0.45,
    "use_hasyv2": True,
    "hasyv2_dir": PROJECT_ROOT / "data" / "hasyv2_digits",
    "hasyv2_weight": 0.20,
    "use_chars74k": True,
    "chars74k_dir": PROJECT_ROOT / "data" / "chars74k_digits",
    "chars74k_weight": 0.10,
    "use_penbased_rendered": True,
    "penbased_dir": PROJECT_ROOT / "data" / "penbased_rendered",
    "penbased_weight": 0.15,
    "use_optical_digits": True,
    "optical_dir": PROJECT_ROOT / "data" / "optical_digits",
    "optical_weight": 0.05,
    "cache_folder_digits": True,
    "robust_affine_degrees": 15.0,
    "robust_noise_std_min": 0.02,
    "robust_noise_std_max": 0.10,
    "robust_blur_prob": 0.30,
    "robust_morph_prob": 0.45,
    "robust_center_jitter": 3,
}
EVAL_CONFIG = {"external_validation_batch_size": 4096, "enable_validation_board": True}
ENSEMBLE_CONFIG = {
    "ensemble_weight_clean": 0.60,
    "ensemble_weight_grid": (0.60, 0.55, 0.50, 0.45, 0.40, 0.35, 0.30),
    "use_tta": True,
    "tta_n": 8,
}
DEBUG_CONFIG = {"verbose": True, "log_interval": 25, "debug_preprocess": True, "debug_preprocess_samples": 16}

base_config = ExperimentConfig(
    project_root=PROJECT_ROOT,
    output_dir=PATH_CONFIG["output_dir"],
    clean_checkpoint_path=CLEAN_CHECKPOINT,
    robust_checkpoint_path=ROBUST_CHECKPOINT,
    seed=42,
    **DATA_CONFIG, **MODEL_CONFIG, **TRAIN_CONFIG, **AUGMENT_CONFIG, **EVAL_CONFIG, **ENSEMBLE_CONFIG, **DEBUG_CONFIG,
)

clean_config = base_config
robust_config = ExperimentConfig(
    **{**base_config.to_dict(), **ROBUST_CONFIG,
       "training_mode": "robust_finetune",
       "clean_checkpoint_path": str(CLEAN_CHECKPOINT),
       "checkpoint_name": "robust_expert_best.pt",
       "learning_rate": ROBUST_CONFIG["fine_tune_lr"],
       "weight_decay": 1e-5,
       "freeze_backbone_first": False,
       "use_local_digits": False,
       "local_digits_weight": 0.0}
)

paths = ensure_project_paths(clean_config)
set_seed(clean_config.seed)
run_results = {"run_info": {}, "stages": [], "training": {}, "validation_board": {}, "holdouts": {}, "ensemble": {}, "prediction": {}, "debug_preprocess": {}}

In [8]:
DO_BENCHMARK_THROUGHPUT = False
BENCHMARK_BATCH_SIZES = (4096, 8192, 12288, 16384)
BENCHMARK_WARMUP_BATCHES = 5
BENCHMARK_TIMED_BATCHES = 30

if DO_BENCHMARK_THROUGHPUT:
    import copy
    from torch import nn
    from src.robust_data import create_robust_finetune_dataloaders
    from src.engine import build_optimizer, make_autocast_context, make_grad_scaler

    benchmark_rows = []
    for candidate_batch_size in BENCHMARK_BATCH_SIZES:
        benchmark_config = copy.deepcopy(robust_config)
        benchmark_config.batch_size = candidate_batch_size
        print(f"\nBenchmark batch_size={candidate_batch_size}")
        try:
            train_loader, _, metadata = create_robust_finetune_dataloaders(benchmark_config)
            model, _ = load_model_from_checkpoint(CLEAN_CHECKPOINT, benchmark_config, DEVICE)
            model.train()
            if DEVICE == "cuda":
                model = model.to(memory_format=torch.channels_last)
                if benchmark_config.compile_model:
                    model = torch.compile(model, mode=benchmark_config.compile_mode)
                torch.cuda.reset_peak_memory_stats()
            criterion = nn.CrossEntropyLoss(label_smoothing=benchmark_config.label_smoothing)
            optimizer = build_optimizer(model, benchmark_config)
            scaler = make_grad_scaler(benchmark_config, DEVICE)
            seen = 0
            timed_batches = 0
            start_time = None
            for batch_index, (images, labels) in enumerate(train_loader):
                if batch_index >= BENCHMARK_WARMUP_BATCHES + BENCHMARK_TIMED_BATCHES:
                    break
                images = images.to(DEVICE, non_blocking=True)
                if DEVICE == "cuda":
                    images = images.contiguous(memory_format=torch.channels_last)
                labels = labels.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with make_autocast_context(benchmark_config, DEVICE):
                    loss = criterion(model(images), labels)
                if scaler.is_enabled():
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()
                if DEVICE == "cuda":
                    torch.cuda.synchronize()
                if batch_index + 1 == BENCHMARK_WARMUP_BATCHES:
                    start_time = time.perf_counter()
                elif batch_index + 1 > BENCHMARK_WARMUP_BATCHES:
                    seen += images.size(0)
                    timed_batches += 1
            elapsed = time.perf_counter() - start_time if start_time else 0.0
            row = {
                "batch_size": candidate_batch_size,
                "timed_batches": timed_batches,
                "samples_per_sec": round(seen / elapsed, 2) if elapsed else None,
                "batches_per_sec": round(timed_batches / elapsed, 3) if elapsed else None,
                "cuda_peak_allocated_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 2) if DEVICE == "cuda" else None,
                "cuda_peak_reserved_gb": round(torch.cuda.max_memory_reserved() / 1024**3, 2) if DEVICE == "cuda" else None,
                "sources": metadata["source_names"],
            }
            benchmark_rows.append(row)
            print(row)
            del model, optimizer, train_loader
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
        except RuntimeError as exc:
            if "out of memory" not in str(exc).lower():
                raise
            print(f"OOM at batch_size={candidate_batch_size}; stop benchmark.")
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            break
    pd.DataFrame(benchmark_rows)
else:
    print("Skipped throughput benchmark. Set DO_BENCHMARK_THROUGHPUT=True to compare batch sizes.")

Skipped throughput benchmark. Set DO_BENCHMARK_THROUGHPUT=True to compare batch sizes.


## 2. Run Card

每个 stage 前都会打印本次运行关键信息。

In [9]:
def stage_card(stage, config, checkpoint=None, datasets=None):
    card = {
        "stage": stage,
        "checkpoint": str(checkpoint) if checkpoint else "",
        "datasets": datasets or [],
        "batch_size": config.batch_size,
        "num_workers": config.num_workers,
        "pin_memory": config.pin_memory,
        "persistent_workers": config.persistent_workers,
        "prefetch_factor": config.prefetch_factor,
        "learning_rate": config.learning_rate,
        "epochs": config.epochs,
        "use_amp": config.use_amp,
        "allow_tf32": config.allow_tf32,
        "compile_model": config.compile_model,
        "compile_mode": config.compile_mode,
        "cache_folder_digits": config.cache_folder_digits,
        "use_tta": config.use_tta,
        "tta_n": config.tta_n,
        "robust_aug_strength": config.robust_aug_strength,
        "output_dir": str(config.resolved_output_dir()),
        "device": DEVICE,
    }
    print("=" * 80)
    print(json.dumps(card, indent=2, ensure_ascii=False))
    return card

def record_stage(stage, status="done", metric=None, artifact=None, notes=None, start_time=None):
    row = {
        "stage": stage,
        "status": status,
        "metric": metric,
        "artifact": str(artifact) if artifact else "",
        "notes": notes or "",
        "duration_sec": round(time.perf_counter() - start_time, 2) if start_time else None,
    }
    run_results["stages"].append(row)
    return row

run_results["run_info"] = {
    "time": datetime.now().isoformat(timespec="seconds"),
    "project_root": str(PROJECT_ROOT),
    "device": DEVICE,
    "clean_checkpoint": str(CLEAN_CHECKPOINT),
    "robust_checkpoint": str(ROBUST_CHECKPOINT),
}
pd.DataFrame([run_results["run_info"]])

,time,project_root,device,clean_checkpoint,robust_checkpoint
0,2026-05-14T13:03:21,e:\ALL\学习\AI导论作业-识别手写数字,cuda,e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\che...,e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\che...


## 3. 可选：数据准备

下载数据通常只需要第一次运行；默认关闭。

In [10]:
if DO_PREPARE_DATA:
    start = time.perf_counter()
    stage_card("prepare_data", clean_config, datasets=["HASYv2", "Chars74K", "Pen-Based"])
    manifest = prepare_finetune_datasets(PROJECT_ROOT, force=False)
    run_results["datasets"] = manifest
    record_stage("prepare_data", artifact=PROJECT_ROOT / "data" / "finetune_datasets_manifest.json", notes=json.dumps(manifest, ensure_ascii=False), start_time=start)
    manifest
else:
    print("Skipped data preparation.")

Skipped data preparation.


## 4. 单独 HPO / 调参

In [11]:
if DO_HPO:
    start = time.perf_counter()
    stage_card("hpo", clean_config, datasets=["MNIST-family small sample"])
    hpo_result = run_hpo(clean_config, n_trials=12, trial_epochs=5, trial_max_samples=12000, device=DEVICE)
    run_results["hpo"] = hpo_result
    metric = hpo_result["best"]["best_val_accuracy"] if hpo_result.get("best") else None
    record_stage("hpo", metric=metric, artifact=hpo_result.get("hpo_dir"), start_time=start)
    pd.DataFrame(hpo_result["rows"])
else:
    print("Skipped HPO.")

Skipped HPO.


## 5. 单独训练 Clean Expert

In [12]:
if DO_TRAIN_CLEAN:
    start = time.perf_counter()
    stage_card("train_clean", clean_config, datasets=["MNIST", "QMNIST", "EMNIST digits", "USPS"])
    train_loader, val_loader = create_dataloaders(clean_config)
    clean_model = build_model(clean_config).to(DEVICE)
    total_params, trainable_params = count_model_parameters(clean_model)
    history = fit(clean_model, train_loader, val_loader, config=clean_config, paths=paths, device=DEVICE)
    checkpoint = paths.checkpoints_dir / clean_config.checkpoint_name
    run_results["training"]["clean"] = history
    record_stage("train_clean", metric=history.get("best_val_accuracy"), artifact=checkpoint, notes=f"params={total_params}, trainable={trainable_params}", start_time=start)
    history
else:
    print("Skipped clean training.")

Skipped clean training.


## 6. 单独训练 Robust Expert

In [13]:
if DO_TRAIN_ROBUST:
    start = time.perf_counter()
    stage_card("train_robust", robust_config, checkpoint=CLEAN_CHECKPOINT, datasets=["MNIST-family", "HASYv2", "Chars74K", "Pen-Based", "Optical(if available)"])
    history = run_robust_finetune(robust_config)
    run_results["training"]["robust"] = history
    metric = history["full_finetune"].get("best_val_accuracy")
    record_stage("train_robust", metric=metric, artifact=ROBUST_CHECKPOINT, start_time=start)
    history
else:
    print("Skipped robust fine-tuning.")

{
  "stage": "train_robust",
  "checkpoint": "e:\\ALL\\学习\\AI导论作业-识别手写数字\\outputs_submission\\checkpoints\\best_model_stat-09987e.pt",
  "datasets": [
    "MNIST-family",
    "HASYv2",
    "Chars74K",
    "Pen-Based",
    "Optical(if available)"
  ],
  "batch_size": 4096,
  "num_workers": 12,
  "pin_memory": true,
  "persistent_workers": true,
  "prefetch_factor": 12,
  "learning_rate": 3e-05,
  "epochs": 60,
  "use_amp": true,
  "allow_tf32": true,
  "compile_model": false,
  "compile_mode": "max-autotune",
  "cache_folder_digits": true,
  "use_tta": true,
  "tta_n": 8,
  "robust_aug_strength": "strong",
  "output_dir": "e:\\ALL\\学习\\AI导论作业-识别手写数字\\outputs_submission",
  "device": "cuda"
}
Robust fine-tuning: preparing run
base checkpoint: e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\checkpoints\best_model_stat-09987e.pt
output_dir: e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission
device: cuda
[load] checkpoint=e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\checkpoints\best_model_stat-09987e.pt d

e:\ALL\学习\AI导论作业-识别手写数字\src\robust_data.py:309: UserWarning: UCI Optical digits 数据目录不存在，已跳过: e:\ALL\学习\AI导论作业-识别手写数字\data\optical_digits
  return _maybe_folder_dataset(


[train] epoch 1/30 batch 1/42 loss=1.0775 acc=0.7522 lr=3e-05 elapsed=130.8s
[train] epoch 1/30 batch 25/42 loss=0.9350 acc=0.7726 lr=3e-05 elapsed=159.8s
[train] epoch 1/30 batch 42/42 loss=0.8886 acc=0.7814 lr=3e-05 elapsed=190.9s
[val] epoch 1/30 batch 1/11 loss=0.2014 acc=0.9980 elapsed=129.6s
[val] epoch 1/30 batch 11/11 loss=0.2996 acc=0.9644 elapsed=134.9s
Epoch 001/030 | train_loss=0.8886 train_acc=0.7814 | val_loss=0.2996 val_acc=0.9644 | best_val_acc=0.9644@1 | lr=2.99178e-05 | best | 325.8s
[train] epoch 2/30 batch 1/42 loss=0.8047 acc=0.7971 lr=2.99178e-05 elapsed=6.3s
[train] epoch 2/30 batch 25/42 loss=0.7636 acc=0.8080 lr=2.99178e-05 elapsed=35.4s
[train] epoch 2/30 batch 42/42 loss=0.7520 acc=0.8111 lr=2.99178e-05 elapsed=55.4s
[val] epoch 2/30 batch 1/11 loss=0.2031 acc=0.9978 elapsed=1.7s
[val] epoch 2/30 batch 11/11 loss=0.2827 acc=0.9680 elapsed=2.4s
Epoch 002/030 | train_loss=0.7520 train_acc=0.8111 | val_loss=0.2827 val_acc=0.9680 | best_val_acc=0.9680@2 | lr=2.96

## 7. 单独评估模型

默认评估 clean checkpoint；如果 robust checkpoint 存在，也会一起评估 robust。

In [14]:
def evaluate_named_checkpoint(name, checkpoint, config):
    checkpoint = Path(checkpoint)
    if not checkpoint.exists():
        row = record_stage(f"evaluate_{name}", status="missing_checkpoint", artifact=checkpoint)
        return row
    start = time.perf_counter()
    stage_card(f"evaluate_{name}", config, checkpoint=checkpoint, datasets=list(config.external_holdout_names))
    model, payload = load_model_from_checkpoint(checkpoint, config, DEVICE)
    board = evaluate_validation_board(model, config, paths.logs_dir, DEVICE, prefix=name)
    holdouts = evaluate_external_holdouts(model, config=config, output_dir=paths.evaluation_dir / f"holdouts_{name}", device=DEVICE)
    run_results["validation_board"][name] = board
    run_results["holdouts"][name] = holdouts
    metric = board["score"]["composite_score"]
    return record_stage(f"evaluate_{name}", metric=metric, artifact=paths.logs_dir / f"validation_board_{name}.json", start_time=start)

if DO_EVALUATE:
    eval_rows = []
    eval_rows.append(evaluate_named_checkpoint("clean", CLEAN_CHECKPOINT, clean_config))
    if ROBUST_CHECKPOINT.exists():
        eval_rows.append(evaluate_named_checkpoint("robust", ROBUST_CHECKPOINT, robust_config))
    pd.DataFrame(eval_rows)
else:
    print("Skipped evaluation.")

{
  "stage": "evaluate_clean",
  "checkpoint": "e:\\ALL\\学习\\AI导论作业-识别手写数字\\outputs_submission\\checkpoints\\best_model_stat-09987e.pt",
  "datasets": [
    "mnist_test",
    "emnist_digits_test",
    "qmnist_test10k"
  ],
  "batch_size": 4096,
  "num_workers": 12,
  "pin_memory": true,
  "persistent_workers": true,
  "prefetch_factor": 12,
  "learning_rate": 0.0008398721379146775,
  "epochs": 60,
  "use_amp": true,
  "allow_tf32": true,
  "compile_model": false,
  "compile_mode": "max-autotune",
  "cache_folder_digits": false,
  "use_tta": true,
  "tta_n": 8,
  "robust_aug_strength": "medium",
  "output_dir": "e:\\ALL\\学习\\AI导论作业-识别手写数字\\outputs_submission",
  "device": "cuda"
}
[load] checkpoint=e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\checkpoints\best_model_stat-09987e.pt device=cuda
[load] model=medium_cnn dropout=0.21672530847241062
[validation-board] building loaders
[validation-board] add val_clean samples=35458
[validation-board] add val_corrupt_lite samples=35458
[validation

## 8. 单独 Ensemble 权重搜索

In [15]:
if DO_ENSEMBLE_SEARCH:
    if not ROBUST_CHECKPOINT.exists():
        raise FileNotFoundError(f"robust checkpoint 不存在: {ROBUST_CHECKPOINT}")
    start = time.perf_counter()
    stage_card("ensemble_search", robust_config, checkpoint=f"{CLEAN_CHECKPOINT} + {ROBUST_CHECKPOINT}", datasets=["validation board"])
    clean_model, _ = load_model_from_checkpoint(CLEAN_CHECKPOINT, clean_config, DEVICE)
    robust_model, _ = load_model_from_checkpoint(ROBUST_CHECKPOINT, robust_config, DEVICE)
    best, rows = search_ensemble_weight(clean_model, robust_model, robust_config, DEVICE, paths.logs_dir)
    run_results["ensemble"] = {"best": best, "rows": rows}
    record_stage("ensemble_search", metric=best.get("composite_score") if best else None, artifact=paths.logs_dir / "ensemble_weight_search.csv", notes=json.dumps(best, ensure_ascii=False), start_time=start)
    pd.DataFrame(rows)
else:
    print("Skipped ensemble search.")

{
  "stage": "ensemble_search",
  "checkpoint": "e:\\ALL\\学习\\AI导论作业-识别手写数字\\outputs_submission\\checkpoints\\best_model_stat-09987e.pt + e:\\ALL\\学习\\AI导论作业-识别手写数字\\outputs_submission\\checkpoints\\robust_expert_best.pt",
  "datasets": [
    "validation board"
  ],
  "batch_size": 4096,
  "num_workers": 12,
  "pin_memory": true,
  "persistent_workers": true,
  "prefetch_factor": 12,
  "learning_rate": 3e-05,
  "epochs": 60,
  "use_amp": true,
  "allow_tf32": true,
  "compile_model": false,
  "compile_mode": "max-autotune",
  "cache_folder_digits": true,
  "use_tta": true,
  "tta_n": 8,
  "robust_aug_strength": "strong",
  "output_dir": "e:\\ALL\\学习\\AI导论作业-识别手写数字\\outputs_submission",
  "device": "cuda"
}
[load] checkpoint=e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\checkpoints\best_model_stat-09987e.pt device=cuda
[load] model=medium_cnn dropout=0.21672530847241062
[load] checkpoint=e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\checkpoints\robust_expert_best.pt device=cuda
[load] model=m

## 9. 单独预测 / 文件夹测试 + preprocess debug

In [16]:
PREDICT_IMAGE_DIR = PATH_CONFIG["exam_image_dir"]
PREDICT_DEBUG_DIR = paths.outputs_dir / "debug_preprocess"

if DO_PREDICT_FOLDER:
    if not ROBUST_CHECKPOINT.exists():
        raise FileNotFoundError(f"robust checkpoint 不存在: {ROBUST_CHECKPOINT}")
    if not PREDICT_IMAGE_DIR.exists():
        raise FileNotFoundError(f"预测图片目录不存在: {PREDICT_IMAGE_DIR}")
    start = time.perf_counter()
    stage_card("predict_folder", robust_config, checkpoint=f"{CLEAN_CHECKPOINT} + {ROBUST_CHECKPOINT}", datasets=[str(PREDICT_IMAGE_DIR)])
    clean_model, _ = load_model_from_checkpoint(CLEAN_CHECKPOINT, clean_config, DEVICE)
    robust_model, _ = load_model_from_checkpoint(ROBUST_CHECKPOINT, robust_config, DEVICE)
    dataset = PredictionImageDataset(PREDICT_IMAGE_DIR, image_size=robust_config.image_size, auto_invert=robust_config.auto_invert)
    loader = DataLoader(dataset, batch_size=robust_config.batch_size, shuffle=False, num_workers=0)
    clean_rows, robust_rows, ensemble_rows = predict_image_folder(clean_model, robust_model, loader, robust_config, DEVICE, clean_weight=robust_config.ensemble_weight_clean)
    write_predictions_csv(clean_rows, paths.predictions_dir / "clean_predictions.csv")
    write_predictions_csv(robust_rows, paths.predictions_dir / "robust_predictions.csv")
    write_predictions_csv(ensemble_rows, paths.predictions_dir / "ensemble_predictions.csv")
    write_predictions_csv(ensemble_rows, paths.outputs_dir / "submission.csv")
    saved_debug = []
    if robust_config.debug_preprocess:
        debug_loader = DataLoader(dataset, batch_size=min(robust_config.debug_preprocess_samples, robust_config.batch_size), shuffle=False, num_workers=0)
        with torch.no_grad():
            for images, filenames in debug_loader:
                clean_prob, robust_prob, ensemble_prob = predict_batch_pair(clean_model, robust_model, images, robust_config, DEVICE, clean_weight=robust_config.ensemble_weight_clean)
                confidence_values, prediction_values = ensemble_prob.max(dim=1)
                for filename, prediction, confidence in zip(filenames, prediction_values.cpu().tolist(), confidence_values.cpu().tolist()):
                    if len(saved_debug) >= robust_config.debug_preprocess_samples:
                        break
                    saved_debug.append(str(save_preprocess_debug_visualization(PREDICT_IMAGE_DIR / filename, PREDICT_DEBUG_DIR, prediction=int(prediction), confidence=float(confidence), image_size=robust_config.image_size, auto_invert=robust_config.auto_invert)))
                break
    run_results["prediction"] = {"submission": str(paths.outputs_dir / "submission.csv"), "num_images": len(ensemble_rows)}
    run_results["debug_preprocess"] = {"dir": str(PREDICT_DEBUG_DIR), "files": saved_debug}
    record_stage("predict_folder", metric=len(ensemble_rows), artifact=paths.outputs_dir / "submission.csv", start_time=start)
    pd.DataFrame(ensemble_rows, columns=["filename", "prediction"]).head()
else:
    print("Skipped folder prediction.")

Skipped folder prediction.


## 10. 本轮运行结果总览

In [17]:
summary_df = pd.DataFrame(run_results["stages"])
if not summary_df.empty:
    summary_path = paths.logs_dir / "notebook_run_results.csv"
    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
    print(f"summary saved: {summary_path}")
summary_df

summary saved: e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\logs\notebook_run_results.csv


,stage,status,metric,artifact,notes,duration_sec
0,train_robust,done,0.985295,e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\che...,,2148.81
1,evaluate_clean,done,0.997861,e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\log...,,375.49
2,evaluate_robust,done,0.989141,e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\log...,,1147.45
3,ensemble_search,done,0.988930,e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\log...,"{""clean_weight"": 0.3, ""composite_score"": 0.988...",2232.68
